In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from catboost import CatBoostRegressor
try:
    from lightgbm import LGBMRegressor
except OSError as e:
    if "libomp" in str(e).lower():
        raise OSError(
            "LightGBM could not load libomp (OpenMP).\n"
            "macOS + Homebrew fix:\n"
            "  brew install libomp\n"
            "Then restart the kernel.\n"
            "Conda fix:\n"
            "  conda install -c conda-forge llvm-openmp\n"
        ) from e
    raise

BASE_DIR = Path("/Users/vishalsankarram/dsci558-project/game_feature_export/artifacts/no_sentiment_with_value")
parquet_path = BASE_DIR / "features_standardized.parquet"

if not parquet_path.exists():
    raise FileNotFoundError(f"Missing file: {parquet_path}")

In [2]:
df = pd.read_parquet(parquet_path)
print("Original shape:", df.shape)

# Drop columns by prefix only
prefix_re = r"(mech_|cat_|price_|rank)"
df = df.loc[:, ~df.columns.str.contains(prefix_re, regex=True)]

# Drop known unnecessary columns if present
cols_to_drop = [
    "categories", "mechanisms", "primary_category", "game_name", "bgg_id"
    "is_expansion", "has_description_embedding",
    "log1p_last_mean", "n_weeks_observed", "log1p_last_min", "log1p_last_max",
    "last_week_spread", "log1p_mean_hist", "log1p_median_weekly_mean",
    "mean_vs_last_ratio", "pct_change_first_to_last", "max_drawdown_mean",
    "weeks_since_last_obs", "calendar_span_weeks", "trailing_null_frac_12w",
    "mean_intraweek_range_last", "mean_intraweek_range_avg", "has_games_csv",
    "stage_c_split"
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print("After drop:", df.shape)

Original shape: (29233, 350)
After drop: (29233, 32)


/var/folders/tq/jtgfkbln5bv4lmlbq2tvnq1h0000gn/T/ipykernel_9098/1781405040.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df.loc[:, ~df.columns.str.contains(prefix_re, regex=True)]


In [3]:
df.columns

Index(['bgg_id', 'mean_embedding', 'description_embedding', 'n_reviews',
       'review_count_feature', 'dispersion', 'median_distance_to_mean',
       'max_distance_to_mean', 'n_unique_reviewers', 'reviews_per_user_mean',
       'reviews_per_user_std', 'value_score_mean', 'value_score_std', 'year',
       'geek_rating', 'avg_rating', 'bayes_average', 'num_voters',
       'is_expansion', 'min_players', 'max_players', 'best_min_players',
       'best_max_players', 'min_playtime', 'max_playtime', 'min_age',
       'complexity', 'coll_share_owns', 'coll_share_wants', 'coll_share_wtb',
       'coll_share_wtt', 'stage_a_split'],
      dtype='object')

In [4]:
SHARE_COLS = ["coll_share_owns", "coll_share_wants", "coll_share_wtb", "coll_share_wtt"]
COUNT_COLS = ["coll_count_owns", "coll_count_wants", "coll_count_wtb", "coll_count_wtt"]

if all(c in df.columns for c in SHARE_COLS):
    TARGET_COLS = SHARE_COLS
elif all(c in df.columns for c in COUNT_COLS):
    TARGET_COLS = COUNT_COLS
else:
    raise ValueError(f"Missing targets. Need either {SHARE_COLS} or {COUNT_COLS}")

print("Targets:", TARGET_COLS)

Targets: ['coll_share_owns', 'coll_share_wants', 'coll_share_wtb', 'coll_share_wtt']


In [5]:
ID_AND_STRING_COLS = {
    "bgg_id",
    "game_name",
    "categories",
    "mechanisms",
    "primary_category",
    "stage_a_split",
    # "stage_c_split",
}

EMBED_COLS = ["mean_embedding", "description_embedding"]

def get_tabular_cols(frame: pd.DataFrame, target_cols: list[str]) -> list[str]:
    exclude = set(ID_AND_STRING_COLS) | set(target_cols) | set(EMBED_COLS)
    cols = []
    for c in frame.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(frame[c]):
            cols.append(c)
    return cols

tabular_cols = get_tabular_cols(df, TARGET_COLS)
print("Tabular columns:", len(tabular_cols))

def stack_embeddings(frame: pd.DataFrame, col: str) -> np.ndarray | None:
    if col not in frame.columns:
        return None
    arrs = [np.asarray(x, dtype=np.float32) for x in frame[col].values]
    return np.vstack(arrs)

def build_X(frame: pd.DataFrame, tabular_cols: list[str]) -> np.ndarray:
    parts = [frame[tabular_cols].to_numpy(dtype=np.float32)]
    for col in EMBED_COLS:
        emb = stack_embeddings(frame, col)
        if emb is not None:
            parts.append(emb)
    return np.hstack(parts)

# Keep only rows with valid targets
df_work = df.loc[np.isfinite(df[TARGET_COLS]).all(axis=1)].copy()

X = build_X(df_work, tabular_cols)
y = df_work[TARGET_COLS].to_numpy(dtype=np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)

Tabular columns: 24
X shape: (28169, 792)
y shape: (28169, 4)


In [6]:
split_vals = df_work["stage_a_split"].astype(str).str.lower()

train_df = df_work.loc[split_vals.eq("train")].copy()
test_df = df_work.loc[split_vals.eq("test")].copy()

# Optional: only use this if "rest" rows are meant to be assigned later
rest_df = df_work.loc[~split_vals.isin(["train", "test"])].copy()
if len(rest_df) > 0:
    rest_train_df, rest_test_df = train_test_split(rest_df, test_size=0.2, random_state=42)
    train_df = pd.concat([train_df, rest_train_df], ignore_index=True)
    test_df = pd.concat([test_df, rest_test_df], ignore_index=True)

X_train = build_X(train_df, tabular_cols)
y_train = train_df[TARGET_COLS].to_numpy(dtype=np.float32)

X_test = build_X(test_df, tabular_cols)
y_test = test_df[TARGET_COLS].to_numpy(dtype=np.float32)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (27427, 792) (27427, 4)
Test: (742, 792) (742, 4)


In [7]:
def evaluate_model(model, X_test, y_test, target_cols, model_name, clip_min=0.0, clip_max=1.0):
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred)

    # Keep predictions in a sensible range for share-like targets
    y_pred = np.clip(y_pred, clip_min, clip_max)

    rows = []
    for i, col in enumerate(target_cols):
        yt = y_test[:, i]
        yp = y_pred[:, i]

        rows.append({
            "model": model_name,
            "target": col,
            "mae": mean_absolute_error(yt, yp),
            "rmse": np.sqrt(mean_squared_error(yt, yp)),
            "r2": r2_score(yt, yp),
        })

    rows.append({
        "model": model_name,
        "target": "overall",
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "r2": r2_score(y_test, y_pred, multioutput="uniform_average"),
    })

    metrics_df = pd.DataFrame(rows)

    pred_df = pd.DataFrame(y_pred, columns=[f"pred_{c}" for c in target_cols])
    true_df = pd.DataFrame(y_test, columns=[f"true_{c}" for c in target_cols])
    out_df = pd.concat([true_df, pred_df], axis=1)
    out_df["model"] = model_name

    return metrics_df, out_df

In [8]:
ridge_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regressor", MultiOutputRegressor(Ridge(alpha=1.0, random_state=42)))
])

ridge_model.fit(X_train, y_train)
ridge_metrics_df, ridge_pred_df = evaluate_model(
    ridge_model, X_test, y_test, TARGET_COLS, model_name="Ridge"
)

In [9]:
cat_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("regressor", MultiOutputRegressor(
        CatBoostRegressor(
            iterations=500,
            learning_rate=0.05,
            depth=6,
            loss_function="RMSE",
            verbose=False,
            random_seed=42
        )
    ))
])

cat_model.fit(X_train, y_train)
cat_metrics_df, cat_pred_df = evaluate_model(
    cat_model, X_test, y_test, TARGET_COLS, model_name="CatBoost"
)

In [10]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.neural_network import MLPRegressor

mlp_base = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regressor", MLPRegressor(
        hidden_layer_sizes=(256, 256, 128, 64),
        activation="relu",
        solver="adam",
        alpha=1e-5,
        batch_size=64,
        learning_rate_init=3e-4,
        max_iter=800,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=25,
        random_state=42,
        verbose=True
    ))
])

mlp_model = TransformedTargetRegressor(
    regressor=mlp_base,
    transformer=StandardScaler()
)

mlp_model.fit(X_train, y_train)

mlp_metrics_df, mlp_pred_df = evaluate_model(
    mlp_model, X_test, y_test, TARGET_COLS, model_name="MLP"
)

Iteration 1, loss = 0.44848787
Validation score: 0.150420
Iteration 2, loss = 0.37166272
Validation score: 0.171491
Iteration 3, loss = 0.31266282
Validation score: 0.163031
Iteration 4, loss = 0.25548827
Validation score: 0.128399
Iteration 5, loss = 0.20436802
Validation score: 0.098548
Iteration 6, loss = 0.15574493
Validation score: 0.079787
Iteration 7, loss = 0.11714728
Validation score: 0.056787
Iteration 8, loss = 0.09159330
Validation score: 0.058789
Iteration 9, loss = 0.07467026
Validation score: 0.058302
Iteration 10, loss = 0.06400524
Validation score: 0.050638
Iteration 11, loss = 0.05739752
Validation score: 0.051235
Iteration 12, loss = 0.05138966
Validation score: 0.044606
Iteration 13, loss = 0.04677786
Validation score: 0.057232
Iteration 14, loss = 0.04317017
Validation score: 0.057250
Iteration 15, loss = 0.04105163
Validation score: 0.057168
Iteration 16, loss = 0.03861526
Validation score: 0.057461
Iteration 17, loss = 0.03588347
Validation score: 0.050784
Iterat

In [11]:
lgbm_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("regressor", MultiOutputRegressor(
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbose=-1
        )
    ))
])

lgbm_model.fit(X_train, y_train)
lgbm_metrics_df, lgbm_pred_df = evaluate_model(
    lgbm_model, X_test, y_test, TARGET_COLS, model_name="LightGBM"
)

/opt/homebrew/Caskroom/miniconda/base/envs/dsci-558-project/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/dsci-558-project/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/dsci-558-project/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/dsci-558-project/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [13]:
rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("regressor", MultiOutputRegressor(
        RandomForestRegressor(
            n_estimators=100,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=42
        )
    ))
])

rf_model.fit(X_train, y_train)
rf_metrics_df, rf_pred_df = evaluate_model(
    rf_model, X_test, y_test, TARGET_COLS, model_name="RandomForest"
)

KeyboardInterrupt: 

In [ ]:
results_df = pd.concat(
    [ridge_metrics_df, cat_metrics_df, mlp_metrics_df, lgbm_metrics_df, rf_metrics_df],
    ignore_index=True
)
results_df = results_df.sort_values(["target", "mae"]).reset_index(drop=True)
print(results_df)

In [ ]:
summary_df = results_df[results_df["target"] == "overall"][["model", "mae", "rmse", "r2"]]
summary_df = summary_df.sort_values("mae").reset_index(drop=True)

print(summary_df)

In [ ]:
results_df

In [ ]:
from pathlib import Path
import joblib

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
joblib.dump(ridge_model, MODELS_DIR / "ridge_model.joblib")
joblib.dump(cat_model, MODELS_DIR / "catboost_model.joblib")
joblib.dump(mlp_model, MODELS_DIR / "mlp_model.joblib")
joblib.dump(lgbm_model, MODELS_DIR / "lightgbm_model.joblib")
joblib.dump(rf_model, MODELS_DIR / "random_forest_model.joblib")

In [ ]:
metadata = {
    "target_cols": TARGET_COLS,
    "feature_count": X_train.shape[1],
    "tabular_cols": tabular_cols,
}

joblib.dump(metadata, MODELS_DIR / "metadata.joblib")

In [ ]:
ridge_model = joblib.load(MODELS_DIR / "ridge_model.joblib")
cat_model = joblib.load(MODELS_DIR / "catboost_model.joblib")
mlp_model = joblib.load(MODELS_DIR / "mlp_model.joblib")
lgbm_model = joblib.load(MODELS_DIR / "lightgbm_model.joblib")
rf_model = joblib.load(MODELS_DIR / "random_forest_model.joblib")

metadata = joblib.load(MODELS_DIR / "metadata.joblib")

In [ ]:
results_df.to_csv(MODELS_DIR / "model_results.csv", index=False)